In [5]:
import scanpy as sc 
import anndata as ad
from collections import Counter
import pandas as pd
import gc
import matplotlib.pyplot as plt
import numpy as np
import os

# Differential metabolites in CXCL10-low Niche_4

In [ ]:
sample_list_sm_with_niche4 = [
    "P42","P39","P43_2","P49","P43_1","P38","P26","P14","P2","P30","P32","P34","P27","P37","P4"
]

In [ ]:
cxcl10_low_niche = ['P49_DBSCAN_7',
                      'P13_DBSCAN_3',
                      'P32_DBSCAN_3',
                      'P32_DBSCAN_1',
                      'P42_DBSCAN_1',
                      'P13_DBSCAN_1',
                      'P26_DBSCAN_3',
                      'P34_DBSCAN_1',
                      'P2_DBSCAN_5',
                      'P42_DBSCAN_2',
                      'P49_DBSCAN_3',
                      'P34_DBSCAN_4',
                      'P4_DBSCAN_1',
                      'P32_DBSCAN_2',
                      'P13_DBSCAN_2',
                      'P14_DBSCAN_3',
                      'P30_DBSCAN_1',
                      'P14_DBSCAN_1',
                      'P34_DBSCAN_2',
                      'P2_DBSCAN_1'
    ]

In [ ]:
for spID in sample_list_sm_with_niche4[0:len(sample_list_sm_with_niche4)]:
    print(spID)
    spID_st = spID.replace('SM', 'ST')
    adata = sc.read_h5ad(f"E:/Projects/BRCA/Intermediate_Data/SM5u/matched_object/{spID}_match_st.h5ad")
    niche4_dbscan = pd.read_csv(f"E:/Projects/BRCA/results/ST/Niche2607/{spID_st}/niche_4_dbscan_cluster_saved.csv")
    niche4_dbscan.set_index('bin', inplace=True)
    niche4_dbscan['cluster_name'] = spID_st + '_' + niche4_dbscan['cluster_name']
    adata.obs['niche_4_dbscan'] = niche4_dbscan['cluster_name'].reindex(adata.obs.index)
    adata.obs['niche_4_dbscan'] = adata.obs['niche_4_dbscan'].fillna('others')
    adata.obs['dem_label'] = 'others'
    adata.obs.loc[adata.obs['niche_4_dbscan'].isin(cxcl10_low_niche), 'dem_label'] = 'cxcl10low_niche'
    if Counter(adata.obs['dem_label'])['cxcl10low_niche'] == 0:
        continue
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)
    sc.tl.rank_genes_groups(adata, "dem_label", method="wilcoxon")
    df  = sc.get.rank_genes_groups_df(adata, group="cxcl10low_niche")
    df.to_csv(f"E:/Projects/BRCA/results/SM/major2607_niche_4_dbscan/niche4_cxcl10low_DBSCAN_dem_{spID}.csv")

# Differential metabolites in CXCL10-high Niche_4

In [ ]:
sample_list_sm_with_niche4 = [
    "P42","P39","P43_2","P49","P43_1","P38","P26","P14","P2","P30","P32","P34","P27","P37","P4"
]

In [ ]:
cxcl10_high_niche = [
    "P49_DBSCAN_9",
    "P42_DBSCAN_3",
    "P30_DBSCAN_2",
    "P43_1_DBSCAN_2",
    "P49_DBSCAN_8",
    "P43_2_DBSCAN_3",
    "P49_DBSCAN_6",
    "P38_DBSCAN_3",
    "P38_DBSCAN_1",
    "P43_2_DBSCAN_4",
    "P43_2_DBSCAN_1",
    "P43_1_DBSCAN_1",
    "P27_DBSCAN_3",
    "P49_DBSCAN_1",
    "P49_DBSCAN_2",
    "P49_DBSCAN_2",
    "P49_DBSCAN_5",
    "P49_DBSCAN_4",
    "P49_DBSCAN_1"
]

In [ ]:
for spID in sample_list_sm_with_niche4[0:len(sample_list_sm_with_niche4)]:
    print(spID)
    spID_st = spID.replace('SM', 'ST')
    adata = sc.read_h5ad(f"E:/Projects/BRCA/Intermediate_Data/SM5u/matched_object/{spID}_match_st.h5ad")
    sc.pp.filter_cells(adata, min_genes=100)
    sc.pp.filter_genes(adata, min_cells=2)
    genes_delete = pd.read_csv('E:/Projects/BRCA/results/SM/filter/noise_metabolites_filtered.csv', index_col=0)
    genes_delete = genes_delete['gene'].astype(str).tolist()
    gene_mask = ~(adata.var_names.isin(genes_delete))
    adata = adata[:,gene_mask]
    
    
    niche4_dbscan = pd.read_csv(f"E:/Projects/BRCA/results/ST/Niche2607/{spID_st}/niche_4_dbscan_cluster_saved.csv")
    niche4_dbscan.set_index('bin', inplace=True)
    niche4_dbscan['cluster_name'] = spID_st + '_' + niche4_dbscan['cluster_name']
    adata.obs['niche_4_dbscan'] = niche4_dbscan['cluster_name'].reindex(adata.obs.index)
    adata.obs['niche_4_dbscan'] = adata.obs['niche_4_dbscan'].fillna('others')
    adata.obs['dem_label'] = 'others'
    adata.obs.loc[adata.obs['niche_4_dbscan'].isin(cxcl10_high_niche), 'dem_label'] = 'cxcl10high'
    if Counter(adata.obs['dem_label'])['cxcl10high'] == 0:
        continue
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)
    sc.tl.rank_genes_groups(adata, "dem_label", method="wilcoxon")
    df  = sc.get.rank_genes_groups_df(adata, group="cxcl10high")
    df.to_csv(f"E:/Projects/BRCA/results/SM/major2607_niche_4_dbscan/niche4_cxcl10high_DBSCAN_dem_{spID}.csv")

# Differential metabolites in plasma cell Niche_11

In [19]:
niche_info = pd.read_csv("E:/Projects/BRCA/results/ST/Niche2607/neighborhood_cluster.csv")

In [ ]:
sample_list_sm_with_niche11 = [
    'P43_2','P43_1','P46','P47','P38','P49','P26','P33','P35','P27'
]

In [ ]:
niche_info_niche11 = niche_info[niche_info['sample'].isin(sample_list_sm_with_niche11)]
niche_info_niche11 = niche_info_niche11[niche_info_niche11['niche'] == 'niche_11']

In [ ]:
for spID in sample_list_sm_with_niche11[0:len(sample_list_sm_with_niche11)]:
    print(spID)
    adata = sc.read_h5ad(f"E:/Projects/BRCA/Intermediate_Data/SM5u/matched_object/{spID}_match_st.h5ad")
    sc.pp.filter_cells(adata, min_genes=100)
    sc.pp.filter_genes(adata, min_cells=2)
    genes_delete = pd.read_csv('E:/Projects/BRCA/results/SM/filter/noise_metabolites_filtered.csv', index_col=0)
    genes_delete = genes_delete['gene'].astype(str).tolist()
    gene_mask = ~(adata.var_names.isin(genes_delete))
    adata = adata[:,gene_mask]
    
    
    adata.obs['niche11'] = 'no'
    adata.obs.loc[adata.obs.index.isin(niche_info_niche11['bin'].tolist()), 'niche11'] = 'niche11'
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)
    sc.tl.rank_genes_groups(adata, "niche11", method="wilcoxon")
    df  = sc.get.rank_genes_groups_df(adata, group="niche11")
    df.to_csv(f"E:/Projects/BRCA/results/SM/Niche11_2607/niche11_dem_{spID}.csv")